In [2]:
import os
import json
import pickle
import warnings
import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import TimeSeriesSplit, HalvingGridSearchCV

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

import xgboost as xgb


In [7]:


# ============================================================
# CONFIG
# ============================================================

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

INPUT_CSV = "../EDA/region_temp_extended.csv"
OUTPUT_DIR = Path("../Outputs/xgBoost")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TODAY = datetime.datetime.today().strftime("%Y-%m-%d")
START = datetime.datetime.now()

RANDOM_STATE = 23
N_JOBS = -1
HORIZON = 1

DATE_COL = "date"
REGION_CANDIDATES = ["name", "region_code"]

TARGET_COL = "next_day"

TRAINING_FEATURES = [
    "next_day",
    "dayofyear",
    "pdtn_doy",
    "de_trend_seas",
    "dts_doyavge",
    "dts_doyvar",
    "yday",
    "IIdays_ago",
    "IIIdays_ago",
    "IVdays_ago",
    "Vdays_ago",
    "VIdays_ago",
    "VIIdays_ago",
    "last_year",
    "last_2year",
    "last_3year",
    "last_4year",
    "last_5year",
    "h1_last_year",
    "h1_last_2years",
    "h1_last_3years",
    "h1_last_4years",
    "h1_last_5years",
    "h1_ly_2days",
    "h1_ly_3day",
    "h1_ly_4day",
    "h1_ly_5day",
    "h1_ly_6day",
    "h1_ly_7day",
    "h1_ly_next_1day",
    "h1_ly_next_2days",
    "h1_ly_next_3days",
    "h1_ly_next_4days",
    "h1_ly_next_5days",
    "h1_ly_next_6days",
    "h1_ly_next_7days",
    "diff_1year",
    "diff_2year",
    "diff_3year",
    "diff_4year",
    "diff_5year",
    "diff_yday",
    "diff_2days",
    "diff_3days",
    "diff_4days",
    "diff_5days",
    "diff_6days",
    "diff_7days",
    "last_7_1day_deltas_mean",
    "last_7_1day_deltas_min",
    "last_7_1day_deltas_max",
]

XGB_GRID = {
    "n_estimators": [5, 50, 80, 100, 150, 200],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.09, 0.10, 0.14, 0.18, 0.20, 0.21],
}

OUTER_CV = TimeSeriesSplit(n_splits=5, test_size=365)
INNER_CV = TimeSeriesSplit(n_splits=3)


# ============================================================
# HELPERS
# ============================================================

def find_region_col(df: pd.DataFrame) -> str:
    for c in REGION_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find region column among {REGION_CANDIDATES}")


def load_data() -> tuple[pd.DataFrame, str]:
    df = pd.read_csv(INPUT_CSV)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])

    region_col = find_region_col(df)

    required = {DATE_COL, region_col, *TRAINING_FEATURES}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.sort_values([region_col, DATE_COL]).reset_index(drop=True)
    return df, region_col


def create_next_day_target(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy().sort_values(DATE_COL)
    g["next_day"] = g["de_trend_seas"].shift(-1)
    return g


def evaluate_fit(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = np.mean(y_pred - y_true)
    r2 = r2_score(y_true, y_pred)
    return {
        "MAE": mae,
        "MAPE": mape,
        "Bias": bias,
        "R2": r2,
        "RMSE": rmse,
    }


# ============================================================
# MAIN
# ============================================================

def main():
    feat_df, region_col = load_data()

    feat_df = (
        feat_df.groupby(region_col, group_keys=False)
        .apply(create_next_day_target)
        .reset_index(drop=True)
    )

    nested_cv_results = []
    best_params_dict = {}
    best_params_rows = []
    fit_metric_rows = []

    imputer = SimpleImputer()
    scaler = StandardScaler()

    for region in feat_df[region_col].dropna().unique():
        print(f"\n=== REGION {region} ===")

        region_df = feat_df[feat_df[region_col] == region].copy().sort_values(DATE_COL)
        train = region_df[TRAINING_FEATURES].dropna(axis=0, how="any").copy()

        if len(train) < 365 * 6:
            print(f"Skipping {region}: too few rows ({len(train)})")
            continue

        X_train = train.drop("next_day", axis="columns")
        X_train = imputer.fit_transform(X_train)
        X_train = scaler.fit_transform(X_train)
        y_train = train["next_day"]

        fold_results = []

        for fold_idx, (train_idx, test_idx) in enumerate(OUTER_CV.split(X_train), start=1):
            print(f"Outer Fold {fold_idx}")

            X_outer_train, X_outer_test = X_train[train_idx], X_train[test_idx]
            y_outer_train, y_outer_test = y_train.iloc[train_idx], y_train.iloc[test_idx]

            grid_search = HalvingGridSearchCV(
                estimator=xgb.XGBRegressor(
                    objective="reg:squarederror",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
                param_grid=XGB_GRID,
                cv=INNER_CV,
                scoring="neg_mean_absolute_percentage_error",
                refit=True,
                n_jobs=N_JOBS,
            )
            grid_search.fit(X_outer_train, y_outer_train)

            best_model = grid_search.best_estimator_
            y_pred = best_model.predict(X_outer_test)

            mape = mean_absolute_percentage_error(y_outer_test, y_pred)
            r2 = r2_score(y_outer_test, y_pred)

            row = {
                "model": "paper_xgboost",
                "region": region,
                "mape": mape,
                "r2": r2,
                "horizon": HORIZON,
                "fold": fold_idx,
            }
            nested_cv_results.append(row)
            fold_results.append(row)

            fold_key = f"{region}_fold{fold_idx}"
            best_params_dict[fold_key] = {
                "model": "paper_xgboost",
                "params": grid_search.best_params_,
            }

            fold_best_params = pd.DataFrame(grid_search.best_params_, index=[fold_key])
            fold_best_params["model"] = "paper_xgboost"
            fold_best_params["region"] = region
            fold_best_params["fold"] = fold_idx
            best_params_rows.append(fold_best_params)

        nested_cv_df = pd.DataFrame(fold_results)
        if nested_cv_df.empty:
            continue

        min_row = nested_cv_df.loc[nested_cv_df["mape"].idxmin()]
        best_fold = f"{min_row['region']}_fold{int(min_row['fold'])}"
        best_params = best_params_dict[best_fold]["params"]

        final_model = xgb.XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=1,
            **best_params,
        )
        final_model.fit(X_train, y_train)

        y_fit = final_model.predict(X_train)
        fit_metrics = evaluate_fit(y_train, y_fit)
        fit_metrics["Model"] = "paper_xgboost"
        fit_metrics["region"] = region
        fit_metric_rows.append(fit_metrics)

        with open(OUTPUT_DIR / f"best_mod_{region}_paper_xgboost.pkl", "wb") as f:
            pickle.dump(final_model, f)

    nested_cv_all = pd.DataFrame(nested_cv_results)
    best_params_df = pd.concat(best_params_rows, axis=0) if best_params_rows else pd.DataFrame()
    fit_metrics_df = pd.DataFrame(fit_metric_rows)

    print("Time taken:", datetime.datetime.now() - START)


if __name__ == "__main__":
    main()


=== REGION 11 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 24 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 27 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 28 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 32 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 44 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 52 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5

=== REGION 53 ===
Outer Fold 1
Outer Fold 2
Outer Fold 3
Outer Fold 4
Outer Fold 5
Time taken: 0:33:40.703859
